# 2. MLP(Multi-Layer Perceptron)

- PyTorch의 `nn.Module`을 사용하여 직접 신경망(MLP)를 설계
- 여러 층으로 구성된 모델의 기울기가 **역전파(Backpropagation)** 알고리즘을 통해 어떻게 계산되는지 이해

## 2-1. 들어가기 전에

### 2-1-1. MLP(Multi-Layer Perceptron)란?

- 선형 계층(Linear Layer) 사이에 `비선형 활성화 함수`를 추가하여 여러 층으로 쌓아 올린 모델
1. **입력층 (Input Layer)**
    - 모델의 `외부 데이터`를 받아들이는 층
    - 입력층의 뉴런(노드)들은 별도의 계산을 수행하지 않고, 입력 신호를 다음 층으로 전달
        - 입력 1 (나이): 20
        - 입력 2 (키): 175
        - 입력 3 (몸무게): 80
        - **통합된 입력 벡터: [20, 175, 80]**
        - 이 MLP의 입력층 뉴런 개수는 3개이다.
2. **은닉층 (Hidden Layer)**
    - 입력 데이터를 비선형적으로 변환하고 데이터의 특징(feature)를 추출하는 층
    - MLP는 `하나 이상의 은닉층`을 가질 수 잇으며, 층의 개수가 많을 수록 `심층 신경망`(DNN)이라고 불림
    - 각 은닉층의 뉴런은 이전 층의 모든 뉴런과 연결되어 있음. 이를 `완전 연결` 구조 라고 함.
        - 입력층 (예:**10개 뉴런**)과 은닉층 1 (예:**20개 뉴런**)을 완전 연결하는 경우를 생각해보자.
        1. **변환의 역할**
            - 10개의 원본 특성이 이 연결을 통해 **새로운 20개의 추상화된 특징**으로 선형 변환
            - 은닉층의 각 뉴런은 입력층의 모든 정보를 종합하여 새로운 표현을 생성
        2. **가중치 개수**
            - **완전 연결**이라는 의미는 **이전 층의 모든 뉴런**이 **현재 층의 모든 뉴런**과 연결된다는 뜻
        3. **결론**
            - 이 연결에 필요한 **가중치(Weights)의 총 개수**는
            $(이전 층 뉴런 수) × (현재 층 뉴런 수)$
        
        → 이 예시에서는 `10 * 20 = 200`개의 가중치가 필요하며, 이는 모델이 학습해야 할 중요한 파라미터
3. **출력층 (Output Layer)**
    - 은닉층을 거쳐 변환된 정보를 바탕으로 `최종 예측 결과`를 출력하는 층
    - 이 층의 뉴런 개수는 모델이 해결하는 문제에 따라 결정됨
        - 이진 분류: 예/아니오 1개 뉴런
        - 10개 범주 분류: 10개 뉴런

## 2-2. 신경망 구현

### 2-2-1. `nn.Module`

- PyTorch에서 모든 신경망 모델의 기초가 되는 모듈
- 신경망은 여러 계층(Layer)과 학습 가능한 파라미터(가중치, 편향)로 구성
- `nn.Module`은 이들을 체계적으로 관리하고 묶어주는 역할

1. `nn.Module`를 상속받은 클래스 구현하기
    1. `__init__`
        - `nn.Linear`: 선형 변환. 입력층 → 은닉층 → 출력층 과정의 변환 과정을 수행
        - `nn.ReLU`: 비선형 변환. ReLU함수는 음수를 0으로 바꾸고 양수는 그대로 둠.
    2. `forward()`
        - 순전파 과정 정의
        - 데이터가 입력으로 들어 왔을 때, 어떤 순서로 거쳐 나갈지 데이터의 흐름을 정의
    3. `self.fc1`: 각 완전 연결층을 의미

In [2]:
import torch
# nn: 신경망 모듈
# nn.Module: 모든 신경망 모듈의 기본 클래스
import torch.nn as nn

# 방법 1: forward 메서드에서 데이터 흐름을 직접 정의
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__() # nn.Module 초기화는 필수입니다.

        # nn.Linear 레이어: 선형 변환
            # 선형 변환: y = xW^T + b (행렬 곱셈 + 편향)
            # input_dim: 입력 특성의 수
            # hidden_dim: 은닉층의 뉴런 수
            # output_dim: 출력 특성의 수
        # 가중치 행렬 W의 크기는 (input_dim, hidden_dim)
        # 편향 벡터 b의 크기는 (hidden_dim,)
        # nn.Linear는 내부적으로 가중치와 편향을 초기화
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # ReLU 활성화 함수: 비선형 변환
            # ReLU(Rectified Linear Unit) 함수는 음수를 0으로 바꾸고 양수는 그대로 둡니다.
            # ReLU(x) = max(0, x)
            # ReLU 함수는 모델에 비선형성을 추가하여 복잡한 패턴을 학습할 수 있게 합니다.
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        # 더 많은 은닉층을 추가하고자 한다면,
        # self.fc3 = nn.Linear(output_dim, another_dim)
        # self.relu2 = nn.ReLU()
        # 와 같이 추가할 수 있음.

    def forward(self, x):
        # 데이터가 레이어를 순서대로 통과하는 과정을 정의
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### 2-2-2. `nn.Sequential`
- 각 계층을 일일히 구현하는 것은 어렵고, 복잡하고, 비효율적임.
- 이를 간결하고 효율적으로 묶어주는 방법

In [13]:
import torch.nn as nn

class SequentialMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=(128, 64), output_dim=3, dropout=0.2):
        super().__init__() 
        h1, h2 = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),

            # 드롭아웃: 과적합 방지
            # 드롭아웃은 학습 과정에서 무작위로 일부 뉴런을 비활성화하여
            # 모델이 특정 뉴런에 과도하게 의존하는 것을 방지
            # p: 비활성화 확률 (0~1 사이 값)
                # 예: p=0.2면 20% 확률로 뉴런 비활성화
            nn.Dropout(p=dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p=dropout),

            nn.Linear(h2, output_dim)
        )

    def forward(self, x):
        # Sequential 컨테이너를 한 번만 호출하면 데이터가 순서대로 흐름
        return self.net(x)

### 2-2-3. 모델 생성 및 순전파 실행
- 오해 금지
    - 현재 모델은 2개의 선형층과 1개의 ReLU 활성화 함수로 구성 됨.
    - 즉 3계층(입력층, 은닉층, 출력층) MLP임
        - 각 Layer 모두 1개씩만 있음.

In [15]:
# 설계도를 바탕으로 실제 모델 객체를 생성
# 입력 차원 20, 은닉층 차원 (10, 10), 출력 차원 3인 MLP 모델 생성
model = SequentialMLP(input_dim=20, hidden_dim=(10, 10), output_dim=3)
print("생성된 모델의 구조:\n", model)

# 모델에 입력할 임의의 더미 데이터(4개 샘플, 20개 특성)를 생성
dummy_input = torch.rand(4, 20)

# 순전파를 실행하여 모델의 예측값(logits)을 얻음
logits = model(dummy_input)

# 예측값의 크기를 확인. (배치 크기, 출력 차원) 형태여야 해.
print("\n예측값(logits)의 크기:", logits.shape)

# 현재는 무작위 값으로 채워진 더미 데이터를 사용했기 때문에
# 예측값을 출력해 보는 것은 큰 의미 없음

생성된 모델의 구조:
 SequentialMLP(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=10, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=10, out_features=10, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=10, out_features=3, bias=True)
  )
)

예측값(logits)의 크기: torch.Size([4, 3])


### 2-2-4. 역전파(Backpropagation)

- 순전파를 통해 얻은 예측값으로 손실을 계산하고, 역전파를 실행
- 각 파라미터 기울기가 계산되는 것을 확인

- 역전파 실행 순서
1. 최종 손실에서 시작
    - 순전파가 종료 된 후, 손실 함수를 통해 최종 손실값을 계산
    - 모델 전체의 `총 오차`
2. 오차를 뒤로 전파
    - 출력층에서 입력층 방향으로 한 계층씩 **거꾸로 전파**
3. 계층별 `책임` 계산
    - 뒤쪽 계층에서 **오차 신호(기울기)**를 전달받은 각 계층은 아래의 두 가지 계산을 실행
    1. 자신의 파라미터(가중치) 책임
        - 이 오차에 자신의 가중치가 얼마나 영향을 미쳤는지 계산해서 `.grad`에 저장
    2. 앞 계층으로 보낼 책임
        - 이 오차를 다시 앞 계층으로 전달하기 위해, 자신에게 들어왔던 `입력 데이터`에 대한 오차 신호를 계산
4. 입력 데이터의 역할
    - 특정 가중치의 책임(기울기)은 순전파 때 그 가중치와 곱해졌던 **입력 데이터의 크기에 비례**
    - 입력값이 컸다면, 그 경로에 있던 가중치의 책임도 더 커지는 셈
5. 학습 준비 완료
    - 위 과정이 입력층까지 반복되면, 모델의 파라미터는 자신의 `.grad` 속성에 `자신이 수정되어야 할 방향과 크기`에 대한 정보를 갖게 됨.

In [8]:
# 재현성을 위해 시드를 고정. 모델의 가중치가 동일하게 초기화 됨
torch.manual_seed(42)

# 모델의 입출력 차원을 데이터에 맞게 수정하여 다시 생성
model = SequentialMLP(input_dim=4, hidden_dim=5, output_dim=2)

# 1. 고정된 입력 데이터와 정답 레이블을 생성
    # (배치 크기=2, 입력 차원=4)
    # 배치 크기: 한 번에 모델에 입력하는 샘플의 수
dummy_input = torch.tensor([
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8]
]).float()

# (배치 크기=2) 이며, 각 샘플의 정답은 0 또는 1
# long(): 정수형 텐서로 변환
dummy_labels = torch.tensor([1, 0]).long()


# 2. 순전파를 통해 예측값(logits) 계산
# forward 메서드는 __call__ 메서드에 의해 자동으로 호출됨
# 즉, model(dummy_input)로 호출하면 내부적으로 model.forward(dummy_input)가 호출됨
logits = model(dummy_input)

# 교차 엔트로피 손실 함수
    # 다중 클래스 분류 문제에 자주 사용
    # 모델의 예측값(로짓)과 실제 정답 레이블 간의 차이를 계산
    # 손실 값이 낮을수록 모델의 예측이 실제 정답에 가까움을 의미
criterion = nn.CrossEntropyLoss()

# 3. 예측값과 정답으로 손실을 계산
loss = criterion(logits, dummy_labels)
print(f"\n계산된 손실(Loss): {loss.item():.4f}") 

# 4. 역전파 실행 전, 기울기 확인 (None)
print("역전파 전, fc1의 기울기:", model.layers[0].weight.grad)

# 5. 역전파를 실행
loss.backward()

# 6. 역전파 실행 후, .grad 속성에 기울기가 채워진 것을 확인
print("역전파 후, fc1의 기울기:", model.layers[0].weight.grad)


계산된 손실(Loss): 0.7958
역전파 전, fc1의 기울기: None
역전파 후, fc1의 기울기: tensor([[ 0.0682,  0.0686,  0.0690,  0.0694],
        [-0.0352, -0.0423, -0.0493, -0.0563],
        [ 0.0853,  0.1023,  0.1194,  0.1364],
        [ 0.0036,  0.0036,  0.0036,  0.0037],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])
